# Rank Retrieval Heads

In [1]:
import os
import json
import numpy as np
from collections import Counter
from datetime import datetime


#### Configuration

In [2]:
EXTRACTION_RUN_NAME = "llama3-8B_instruct_run_2026-03-21_01-10-12" # Update this to your extraction run
EXTRACTION_DIR = f"../../../../data/retrieval_heads/01_extractions/{EXTRACTION_RUN_NAME}"
RAW_TENSORS_DIR = os.path.join(EXTRACTION_DIR, "raw_tensors")

# Load source model from extraction meta
with open(os.path.join(EXTRACTION_DIR, "meta.json"), "r") as f:
    extraction_meta = json.load(f)
SOURCE_MODEL = extraction_meta.get("model_id")

TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RANKING_RUN_NAME = f"rank_{TIMESTAMP}"
RANKING_OUTPUT_DIR = f"../../../../data/retrieval_heads/02_rankings/{RANKING_RUN_NAME}"
os.makedirs(RANKING_OUTPUT_DIR, exist_ok=True)

TOP_K_HEADS = 128
GLOBAL_TOP_K_HEADS = 128
MIN_TASKS_SHARED = 7

print(f"Source Model: {SOURCE_MODEL}")
print(f"Reading from: {RAW_TENSORS_DIR}")
print(f"Writing to: {RANKING_OUTPUT_DIR}")


Source Model: meta-llama/Meta-Llama-3-8B-Instruct
Reading from: ../../../../data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12\raw_tensors
Writing to: ../../../../data/retrieval_heads/02_rankings/rank_2026-03-20_21-55-01


#### Load Tensors and Calculate Task Mean Scores

In [4]:
tasks = [
    "registrant_name", "headquarters_city", "headquarters_state",
    "incorporation_state", "incorporation_year", "employees_count_total",
    "holder_record_amount", "employees_count_full_time", "ceo_lastname"
]

task_mean_scores = {}
all_tensors = []

for task_id in tasks:
    files = [f for f in os.listdir(RAW_TENSORS_DIR) if f.endswith(f"_{task_id}.npy")]
    if not files:
        print(f"WARNING: no files found for task '{task_id}' — skipping")
        continue
    loaded = [np.load(os.path.join(RAW_TENSORS_DIR, f)) for f in files]
    stacked = np.stack(loaded)
    task_mean_scores[task_id] = stacked.mean(axis=0)
    all_tensors.extend(loaded)
    print(f"{task_id:<35} {len(files):>4} samples -> mean shape {task_mean_scores[task_id].shape}")

print(f"\n{len(task_mean_scores)} tasks aggregated.")


registrant_name                      156 samples -> mean shape (32, 32)
headquarters_city                    110 samples -> mean shape (32, 32)
headquarters_state                   107 samples -> mean shape (32, 32)
incorporation_state                  120 samples -> mean shape (32, 32)
incorporation_year                   123 samples -> mean shape (32, 32)
employees_count_total                114 samples -> mean shape (32, 32)
holder_record_amount                 118 samples -> mean shape (32, 32)
employees_count_full_time             36 samples -> mean shape (32, 32)
ceo_lastname                         131 samples -> mean shape (32, 32)

9 tasks aggregated.


#### Calculate Global Mean Scores (across ALL documents)

In [5]:
global_mean_scores = np.stack(all_tensors).mean(axis=0)
print(f"Global mean scores shape: {global_mean_scores.shape} across {len(all_tensors)} total documents.")


Global mean scores shape: (32, 32) across 1015 total documents.


#### 1. Per-Task Top Heads

In [7]:
task_top_heads = {}
for task_id, mean_scores in task_mean_scores.items():
    flat = mean_scores.flatten()
    top_flat = np.argsort(flat)[::-1][:TOP_K_HEADS]
    num_heads = mean_scores.shape[1]
    heads = [
        {
            "layer": int(idx // num_heads),
            "head": int(idx % num_heads),
            "score": float(flat[idx]),
        }
        for idx in top_flat
    ]
    task_top_heads[task_id] = heads


#### 2. Shared Heads (Intersection)

In [8]:
all_top_heads = []
for heads in task_top_heads.values():
    all_top_heads.extend([(h["layer"], h["head"]) for h in heads])

head_counts = Counter(all_top_heads)
shared_head_pairs = [head for head, count in head_counts.items() if count >= MIN_TASKS_SHARED]

shared_heads = []
for layer, head in shared_head_pairs:
    freq = head_counts[(layer, head)]
    avg_score = float(np.mean([
        task_mean_scores[tid][layer, head]
        for tid in task_mean_scores
    ]))
    shared_heads.append({
        "layer": layer,
        "head": head,
        "avg_score": avg_score,
        "task_frequency": freq
    })

shared_heads.sort(key=lambda x: (x["task_frequency"], x["avg_score"]), reverse=True)
print(f"Found {len(shared_heads)} shared heads (min {MIN_TASKS_SHARED} tasks).")


Found 102 shared heads (min 7 tasks).


#### 3. Global Heads (Top overall across all data)

In [9]:
flat_global = global_mean_scores.flatten()
top_flat_global = np.argsort(flat_global)[::-1][:GLOBAL_TOP_K_HEADS]
num_heads = global_mean_scores.shape[1]

global_heads = [
    {
        "layer": int(idx // num_heads),
        "head": int(idx % num_heads),
        "score": float(flat_global[idx]),
    }
    for idx in top_flat_global
]

print(f"Identified top {len(global_heads)} global heads.")


Identified top 128 global heads.


#### Export Results

In [10]:
output = {
    "per_task_top_k": TOP_K_HEADS,
    "tasks": task_top_heads,
    "shared_heads": shared_heads,
    "global_heads": global_heads
}
out_path = os.path.join(RANKING_OUTPUT_DIR, "ranked_heads.json")

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

meta = {
    "source_model": SOURCE_MODEL,
    "source_extraction": EXTRACTION_RUN_NAME,
    "timestamp": TIMESTAMP,
    "top_k_heads": TOP_K_HEADS,
    "global_top_k_heads": GLOBAL_TOP_K_HEADS,
    "min_tasks_shared": MIN_TASKS_SHARED
}
with open(os.path.join(RANKING_OUTPUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved rankings to {RANKING_OUTPUT_DIR}")


Saved rankings to ../../../../data/retrieval_heads/02_rankings/rank_2026-03-20_21-55-01
